# SmartTraffic — RB-TFormer
## Robustness-Balanced Transformer via Validation-Selected Checkpoint Interpolation

This notebook performs a **final model-selection experiment** without retraining the vision pipeline and without selecting any parameter on the test set.

### Models shown in outputs

- Transformer
- ReliabilityConcat
- VRA-TFormer
- RR-TFormer
- **RB-TFormer**

### Core idea

Transformer provides the strongest absolute robustness in the completed experiment, while RR-TFormer exhibits strong corruption regularization but loses clean accuracy.

Because both Transformer and RR-TFormer share the same forecasting-backbone architecture, RB-TFormer interpolates their backbone checkpoints:

\[
\theta_{\alpha}
=
(1-\alpha)\theta_{\mathrm{Transformer}}
+
\alpha\theta_{\mathrm{RR}}
\]

where

\[
\alpha \in \{0.0,0.1,\ldots,1.0\}.
\]

### Strict selection protocol

**Alpha is selected using validation data only.**

Selection uses:
1. clean validation MAE, and
2. synthetic-corruption validation MAE.

The real clean test set and the real degraded-cache stress test are not used for alpha selection.

After alpha is locked, RB-TFormer is evaluated once on:
- chronological clean test windows,
- the existing paired real degraded-cache stress test.

### No expensive reprocessing

This notebook does NOT:
- download datasets,
- run YOLO,
- run ByteTrack,
- regenerate degradation video,
- retrain the forecasting models.

It only loads cached parquet and existing checkpoints.

In [ ]:
# ============================================================
# 0. SETUP
# ============================================================
%pip -q install pyarrow openpyxl scipy statsmodels scikit-learn tqdm tabulate

from google.colab import drive
drive.mount('/content/drive')

import os, math, json, random, zipfile, warnings, re
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt

from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from scipy.stats import wilcoxon
from statsmodels.stats.multitest import multipletests

warnings.filterwarnings("ignore")

print("Drive mounted. No video processing will be performed.")

In [ ]:
# ============================================================
# 1. CONFIG + PATHS
# ============================================================
CFG = {
    "SEED": 42,
    "PROJECT_NAME": "SmartTraffic_VRA_TFormer_IJIES",

    "BIN_SEC": 5,
    "LOOKBACK_STEPS": 60,
    "FORECAST_BLOCK_STEPS": 6,
    "N_HORIZONS": 4,
    "TRAIN_FRAC": 0.60,
    "VAL_FRAC": 0.20,

    "D_MODEL": 64,
    "N_HEAD": 4,
    "N_LAYERS": 2,
    "DROPOUT": 0.10,

    "BATCH_SIZE": 128,
    "ROBUST_LOOKBACK_STEPS": 24,

    # Fixed candidate set declared before evaluation.
    "ALPHAS": [0.0,0.1,0.2,0.3,0.4,0.5,0.6,0.7,0.8,0.9,1.0],

    # Validation selection weights.
    # Clean and synthetic-corruption validation errors receive equal weight
    # after normalization to the Transformer reference.
    "W_VAL_CLEAN": 0.50,
    "W_VAL_CORRUPT": 0.50,

    "FIG_DPI": 400,
}

SEED = CFG["SEED"]
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("DEVICE:", DEVICE)

ROOT = Path("/content/drive/MyDrive") / CFG["PROJECT_NAME"]

D = {
    "root": ROOT,
    "cache": ROOT / "cache",
    "clean": ROOT / "cache" / "aicity_clean",
    "degraded": ROOT / "cache" / "aicity_degraded",
    "models": ROOT / "models",
    "rr": ROOT / "rr",
    "rb": ROOT / "rb",
}

D["rb"].mkdir(parents=True, exist_ok=True)

if not ROOT.exists():
    raise FileNotFoundError(f"Project folder not found: {ROOT}")

print("Project root:", ROOT)

In [ ]:
# ============================================================
# 2. LOAD CACHED DATA
# ============================================================
def load_cache(aggregate_path, parts_root):
    aggregate_path = Path(aggregate_path)

    if aggregate_path.exists():
        print("[cache]", aggregate_path.name)
        return pd.read_parquet(aggregate_path)

    parts = sorted(Path(parts_root).rglob("*.parquet"))

    if not parts:
        raise RuntimeError(f"No parquet files found under {parts_root}")

    df = pd.concat(
        [pd.read_parquet(p) for p in parts],
        ignore_index=True
    )

    df.to_parquet(aggregate_path, index=False)
    return df

clean_df = load_cache(
    D["cache"] / "aicity_clean_all.parquet",
    D["clean"]
)

degraded_df = load_cache(
    D["cache"] / "aicity_degraded_all.parquet",
    D["degraded"]
)

clean_df = clean_df.sort_values(
    ["sequence","bin_idx"]
).reset_index(drop=True)

degraded_df = degraded_df.sort_values(
    ["condition","severity","sequence","bin_idx"]
).reset_index(drop=True)

TRAFFIC_COLS = ["car","motorcycle","bus","truck","total"]

REL_COLS = [
    "mean_conf",
    "track_continuity",
    "blur_log",
    "illumination_quality",
    "occlusion_quality",
    "frame_availability",
]

print("Clean sequences:", clean_df["sequence"].nunique())
print("Degraded sequences:", degraded_df["sequence"].nunique())

In [ ]:
# ============================================================
# 3. RECONSTRUCT EXACT SPLIT + SCALERS
# ============================================================
for c in REL_COLS:
    clean_df[c] = clean_df.groupby("sequence")[c].transform(
        lambda s: s.interpolate(limit_direction="both")
    )

bounds = {}

for seq,g in clean_df.groupby("sequence"):
    n = len(g)
    tr = int(n * CFG["TRAIN_FRAC"])
    va = int(n * (CFG["TRAIN_FRAC"] + CFG["VAL_FRAC"]))
    bounds[seq] = (tr,va,n)

train_row_mask = np.zeros(len(clean_df),dtype=bool)

for seq,idx in clean_df.groupby("sequence").groups.items():
    ids = list(idx)
    tr = bounds[seq][0]
    train_row_mask[ids[:tr]] = True

rel_median = clean_df.loc[
    train_row_mask,
    REL_COLS
].median()

for c in REL_COLS:
    clean_df[c] = clean_df[c].fillna(rel_median[c])

x_scaler = StandardScaler().fit(
    clean_df.loc[
        train_row_mask,
        TRAFFIC_COLS
    ]
)

q_scaler = StandardScaler().fit(
    clean_df.loc[
        train_row_mask,
        REL_COLS
    ]
)

L = CFG["LOOKBACK_STEPS"]
B = CFG["FORECAST_BLOCK_STEPS"]
H = CFG["N_HORIZONS"]
FUTURE = B*H

samples = []

for seq,g in clean_df.groupby("sequence",sort=False):
    g = g.reset_index(drop=True)
    tr,va,n = bounds[seq]

    X_raw = g[TRAFFIC_COLS].values.astype(np.float32)
    X = x_scaler.transform(g[TRAFFIC_COLS]).astype(np.float32)
    Q = q_scaler.transform(g[REL_COLS]).astype(np.float32)

    y_series = g["total"].values.astype(float)

    for t in range(L,n-FUTURE+1):
        y_raw = np.array([
            y_series[t+h*B:t+(h+1)*B].sum()
            for h in range(H)
        ],dtype=np.float32)

        target_end = t+FUTURE

        if target_end <= tr:
            split = "train"
        elif t >= tr and target_end <= va:
            split = "val"
        elif t >= va:
            split = "test"
        else:
            continue

        samples.append({
            "sequence":seq,
            "split":split,
            "t":int(t),
            "x":X[t-L:t],
            "q":Q[t-L:t],
            "x_raw":X_raw[t-L:t],
            "y_raw":y_raw,
        })

train_targets = np.vstack([
    r["y_raw"]
    for r in samples
    if r["split"]=="train"
])

y_scaler = StandardScaler().fit(
    train_targets.reshape(-1,1)
)

for r in samples:
    r["y"] = (
        y_scaler
        .transform(r["y_raw"].reshape(-1,1))
        .reshape(-1)
        .astype(np.float32)
    )

split_summary = pd.DataFrame({
    "split":["train","val","test"],
    "windows":[
        sum(r["split"]==s for r in samples)
        for s in ["train","val","test"]
    ]
})

display(split_summary)

expected = {"train":798,"val":294,"test":294}
observed = dict(zip(split_summary["split"],split_summary["windows"]))

if observed != expected:
    raise RuntimeError(
        f"Split mismatch. Expected {expected}, got {observed}."
    )

print("✅ Exact split reproduced.")

In [ ]:
# ============================================================
# 4. MODEL DEFINITIONS
# ============================================================
NT = len(TRAFFIC_COLS)
NR = len(REL_COLS)
NO = CFG["N_HORIZONS"]
DM = CFG["D_MODEL"]

class PositionalEncoding(nn.Module):
    def __init__(self,d_model,max_len=2048):
        super().__init__()

        pe = torch.zeros(max_len,d_model)
        pos = torch.arange(max_len).unsqueeze(1).float()

        div = torch.exp(
            torch.arange(0,d_model,2).float()
            * (-math.log(10000.0)/d_model)
        )

        pe[:,0::2] = torch.sin(pos*div)
        pe[:,1::2] = torch.cos(pos*div)

        self.register_buffer(
            "pe",
            pe.unsqueeze(0),
            persistent=False
        )

    def forward(self,x):
        return x + self.pe[:,:x.size(1)]

class TransformerModel(nn.Module):
    def __init__(self,input_dim=NT):
        super().__init__()

        self.p = nn.Linear(input_dim,DM)
        self.pos = PositionalEncoding(DM)

        layer = nn.TransformerEncoderLayer(
            DM,
            CFG["N_HEAD"],
            DM*4,
            CFG["DROPOUT"],
            batch_first=True,
            norm_first=True
        )

        self.e = nn.TransformerEncoder(
            layer,
            CFG["N_LAYERS"]
        )

        self.h = nn.Sequential(
            nn.LayerNorm(DM),
            nn.Linear(DM,NO)
        )

    def encode(self,x):
        return self.e(
            self.pos(
                self.p(x)
            )
        )

    def forward(self,x,q=None):
        z = self.encode(x)
        return self.h(z.mean(dim=1))

class ReliabilityConcatModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.m = TransformerModel(NT+NR)

    def forward(self,x,q=None):
        return self.m(
            torch.cat([x,q],dim=-1)
        )

class ReliabilityAttentionLayer(nn.Module):
    def __init__(self):
        super().__init__()

        self.nh = CFG["N_HEAD"]

        self.att = nn.MultiheadAttention(
            DM,
            self.nh,
            dropout=CFG["DROPOUT"],
            batch_first=True
        )

        self.n1 = nn.LayerNorm(DM)
        self.n2 = nn.LayerNorm(DM)

        self.ff = nn.Sequential(
            nn.Linear(DM,DM*4),
            nn.GELU(),
            nn.Dropout(CFG["DROPOUT"]),
            nn.Linear(DM*4,DM)
        )

    def forward(self,x,reliability=None,lam=None):
        mask=None

        if reliability is not None and lam is not None:
            b0,l0,_ = reliability.shape

            key_bias = (
                lam
                * torch.log(
                    reliability.squeeze(-1).clamp_min(1e-5)
                )
            )

            mask = (
                key_bias[:,None,:]
                .expand(b0,l0,l0)
                .repeat_interleave(self.nh,dim=0)
            )

        h = self.n1(x)

        a,_ = self.att(
            h,h,h,
            attn_mask=mask,
            need_weights=False
        )

        x = x+a

        return x + self.ff(self.n2(x))

class VRATFormer(nn.Module):
    def __init__(self):
        super().__init__()

        self.obs = nn.Sequential(
            nn.Linear(NT,DM),
            nn.GELU()
        )

        self.rel = nn.Sequential(
            nn.Linear(NR,DM//2),
            nn.GELU()
        )

        self.gate = nn.Sequential(
            nn.Linear(DM//2,1),
            nn.Sigmoid()
        )

        self.ctx = nn.GRU(
            NT,DM,
            batch_first=True
        )

        self.pos = PositionalEncoding(DM)

        self.layers = nn.ModuleList([
            ReliabilityAttentionLayer()
            for _ in range(CFG["N_LAYERS"])
        ])

        self.rawlam = nn.Parameter(torch.tensor(0.0))

        self.head = nn.Sequential(
            nn.LayerNorm(DM),
            nn.Linear(DM,NO)
        )

    def forward(self,x,q=None):
        obs = self.obs(x)
        ctx,_ = self.ctx(x)

        r = self.gate(self.rel(q))

        z = r*obs + (1-r)*ctx
        z = self.pos(z)

        lam = F.softplus(self.rawlam)

        for layer in self.layers:
            z = layer(z,r,lam)

        return self.head(z.mean(dim=1))

class RRTFormer(nn.Module):
    def __init__(self,transformer_state):
        super().__init__()

        self.backbone = TransformerModel()
        self.backbone.load_state_dict(transformer_state)

        self.reconstruction_head = nn.Sequential(
            nn.Linear(DM,DM),
            nn.GELU(),
            nn.Linear(DM,NT)
        )

        self.reliability_head = nn.Sequential(
            nn.Linear(DM,32),
            nn.GELU(),
            nn.Linear(32,1),
            nn.Sigmoid()
        )

    def forward(self,x,return_aux=False):
        z = self.backbone.encode(x)
        pred = self.backbone.h(z.mean(dim=1))

        if not return_aux:
            return pred

        return {
            "pred":pred,
            "reconstructed":self.reconstruction_head(z),
            "reliability":self.reliability_head(z),
        }

def safe_model_name(name):
    return re.sub(
        r"[^A-Za-z0-9_-]+",
        "_",
        name
    )

def load_state(path):
    try:
        return torch.load(
            path,
            map_location=DEVICE,
            weights_only=True
        )
    except TypeError:
        return torch.load(
            path,
            map_location=DEVICE
        )

TRANSFORMER_CKPT = D["models"] / "Transformer.pt"
CONCAT_CKPT = D["models"] / "Transformer_ReliabilityConcat.pt"
VRA_CKPT = D["models"] / "VRA-TFormer.pt"
RR_CKPT = D["rr"] / "RR-TFormer_FINAL.pt"

for name,path in {
    "Transformer":TRANSFORMER_CKPT,
    "ReliabilityConcat":CONCAT_CKPT,
    "VRA-TFormer":VRA_CKPT,
    "RR-TFormer":RR_CKPT,
}.items():
    print(name,"✅" if path.exists() else "❌",path)

if not all(
    p.exists()
    for p in [
        TRANSFORMER_CKPT,
        CONCAT_CKPT,
        VRA_CKPT,
        RR_CKPT,
    ]
):
    raise RuntimeError(
        "One or more required checkpoints are missing."
    )

transformer = TransformerModel().to(DEVICE)
reliability_concat = ReliabilityConcatModel().to(DEVICE)
vra = VRATFormer().to(DEVICE)

transformer.load_state_dict(
    load_state(TRANSFORMER_CKPT)
)

reliability_concat.load_state_dict(
    load_state(CONCAT_CKPT)
)

vra.load_state_dict(
    load_state(VRA_CKPT)
)

rr_container = RRTFormer(
    load_state(TRANSFORMER_CKPT)
).to(DEVICE)

rr_state = torch.load(
    RR_CKPT,
    map_location=DEVICE
)

if isinstance(rr_state,dict) and "model" in rr_state:
    rr_container.load_state_dict(rr_state["model"])
else:
    rr_container.load_state_dict(rr_state)

rr_backbone_state = {
    k.replace("backbone.","",1):v
    for k,v in rr_container.state_dict().items()
    if k.startswith("backbone.")
}

transformer_state = load_state(
    TRANSFORMER_CKPT
)

print("✅ All mature-model checkpoints loaded.")

In [ ]:
# ============================================================
# 5. FIXED SYNTHETIC VALIDATION CORRUPTION
# VALIDATION ONLY — USED FOR ALPHA SELECTION
# ============================================================
def synth_val_corrupt(x_raw,rng):
    x = x_raw.copy().astype(np.float32)

    severity = float(rng.uniform(0.25,0.75))

    condition = str(
        rng.choice([
            "blur",
            "lowlight",
            "occlusion",
            "frame_drop",
        ])
    )

    if condition == "blur":
        x[:,:4] *= (1.0 - 0.32*severity)

    elif condition == "lowlight":
        x[:,:4] *= (1.0 - 0.28*severity)

    elif condition == "occlusion":
        x[:,:4] *= (1.0 - 0.35*severity)

        mask = (
            rng.random(len(x))
            < (0.08 + 0.16*severity)
        )

        x[mask,:4] *= rng.uniform(
            0.25,
            max(0.30,1-severity)
        )

    elif condition == "frame_drop":
        drop_p = min(0.45,0.50*severity)

        mask = (
            rng.random(len(x))
            < drop_p
        )

        x[mask,:4] *= rng.uniform(
            0.0,
            max(0.15,1-severity)
        )

        x[~mask,:4] *= (1.0 - 0.08*severity)

    noise_sd = 0.02 + 0.05*severity

    x[:,:4] *= np.clip(
        rng.normal(
            1.0,
            noise_sd,
            size=x[:,:4].shape
        ),
        0.0,
        None
    )

    x[:,:4] = np.clip(
        x[:,:4],
        0.0,
        None
    )

    x[:,4] = x[:,:4].sum(axis=1)

    return x_scaler.transform(x).astype(np.float32)

val_rows = [
    r
    for r in samples
    if r["split"]=="val"
]

val_clean = []
val_corrupt = []

for i,r in enumerate(val_rows):
    rng = np.random.default_rng(
        SEED + 100000 + i*1237
    )

    val_clean.append({
        "x":r["x"],
        "y":r["y"],
    })

    val_corrupt.append({
        "x":synth_val_corrupt(
            r["x_raw"],
            rng
        ),
        "y":r["y"],
    })

class XOnlyDS(Dataset):
    def __init__(self,rows):
        self.rows=rows

    def __len__(self):
        return len(self.rows)

    def __getitem__(self,i):
        r=self.rows[i]

        return (
            torch.tensor(r["x"],dtype=torch.float32),
            torch.tensor(r["y"],dtype=torch.float32),
        )

val_clean_loader = DataLoader(
    XOnlyDS(val_clean),
    batch_size=CFG["BATCH_SIZE"],
    shuffle=False
)

val_corrupt_loader = DataLoader(
    XOnlyDS(val_corrupt),
    batch_size=CFG["BATCH_SIZE"],
    shuffle=False
)

print("Validation windows:",len(val_clean))

In [ ]:
# ============================================================
# 6. CHECKPOINT INTERPOLATION + VALIDATION SELECTION
# ============================================================
def interpolate_state(
    state_a,
    state_b,
    alpha
):
    out={}

    for k in state_a.keys():
        a=state_a[k]
        b=state_b[k]

        if torch.is_floating_point(a):
            out[k] = (
                (1-alpha)*a
                + alpha*b
            )
        else:
            out[k] = a.clone()

    return out

@torch.no_grad()
def predict_xonly(model,loader):
    model.eval()

    ys=[]
    ps=[]

    for x,y in loader:
        x=x.to(DEVICE)
        p=model(x)

        ys.append(
            y.numpy()
        )

        ps.append(
            p.cpu().numpy()
        )

    y=np.concatenate(ys)
    p=np.concatenate(ps)

    y_raw=(
        y_scaler
        .inverse_transform(
            y.reshape(-1,1)
        )
        .reshape(y.shape)
    )

    p_raw=(
        y_scaler
        .inverse_transform(
            p.reshape(-1,1)
        )
        .reshape(p.shape)
    )

    return y_raw,p_raw

def global_mae(y,p):
    return float(
        np.abs(y-p).mean()
    )

# Transformer validation reference for normalization.
ref_model = TransformerModel().to(DEVICE)
ref_model.load_state_dict(transformer_state)

yc_ref,pc_ref = predict_xonly(
    ref_model,
    val_clean_loader
)

yd_ref,pd_ref = predict_xonly(
    ref_model,
    val_corrupt_loader
)

ref_clean_mae = global_mae(yc_ref,pc_ref)
ref_corrupt_mae = global_mae(yd_ref,pd_ref)

print("Transformer validation clean MAE:",ref_clean_mae)
print("Transformer synthetic-corruption validation MAE:",ref_corrupt_mae)

selection_rows=[]

for alpha in CFG["ALPHAS"]:

    model = TransformerModel().to(DEVICE)

    state = interpolate_state(
        transformer_state,
        rr_backbone_state,
        alpha
    )

    model.load_state_dict(state)

    yc,pc = predict_xonly(
        model,
        val_clean_loader
    )

    yd,pred_corrupt = predict_xonly(
        model,
        val_corrupt_loader
    )

    mae_clean = global_mae(yc,pc)
    mae_corrupt = global_mae(yd,pred_corrupt)

    clean_ratio = (
        mae_clean
        / max(ref_clean_mae,1e-9)
    )

    corrupt_ratio = (
        mae_corrupt
        / max(ref_corrupt_mae,1e-9)
    )

    score = (
        CFG["W_VAL_CLEAN"]*clean_ratio
        + CFG["W_VAL_CORRUPT"]*corrupt_ratio
    )

    selection_rows.append({
        "alpha":alpha,
        "Validation_Clean_MAE":mae_clean,
        "Validation_Corrupt_MAE":mae_corrupt,
        "Clean_Ratio_to_Transformer":clean_ratio,
        "Corrupt_Ratio_to_Transformer":corrupt_ratio,
        "Selection_Score":score,
    })

selection = pd.DataFrame(selection_rows)

# Strictly validation-selected alpha.
best_row = selection.sort_values(
    ["Selection_Score","Validation_Clean_MAE","alpha"]
).iloc[0]

SELECTED_ALPHA = float(best_row["alpha"])

display(selection)

print("\n===================================")
print("SELECTED ALPHA:",SELECTED_ALPHA)
print("Selection score:",best_row["Selection_Score"])
print("===================================")

selection.to_csv(
    D["rb"]/"RB_alpha_validation_selection.csv",
    index=False
)

In [ ]:
# ============================================================
# 7. BUILD LOCKED RB-TFORMER
# ============================================================
rb = TransformerModel().to(DEVICE)

rb_state = interpolate_state(
    transformer_state,
    rr_backbone_state,
    SELECTED_ALPHA
)

rb.load_state_dict(rb_state)

torch.save(
    {
        "model":rb.state_dict(),
        "selected_alpha":SELECTED_ALPHA,
        "selection_protocol":"validation_clean_plus_synthetic_corruption",
        "config":CFG,
    },
    D["rb"]/"RB-TFormer_FINAL.pt"
)

print(
    "RB-TFormer locked with alpha =",
    SELECTED_ALPHA
)

In [ ]:
# ============================================================
# 8. CLEAN TEST EVALUATION
# ============================================================
def inverse_target(a):
    a=np.asarray(a)
    shape=a.shape

    return (
        y_scaler
        .inverse_transform(
            a.reshape(-1,1)
        )
        .reshape(shape)
    )

class FullDS(Dataset):
    def __init__(self,rows,split):
        self.rows=[
            r
            for r in rows
            if r["split"]==split
        ]

    def __len__(self):
        return len(self.rows)

    def __getitem__(self,i):
        r=self.rows[i]

        return (
            torch.tensor(r["x"],dtype=torch.float32),
            torch.tensor(r["q"],dtype=torch.float32),
            torch.tensor(r["y"],dtype=torch.float32),
            r["sequence"],
            int(r["t"]),
        )

test_loader=DataLoader(
    FullDS(samples,"test"),
    batch_size=CFG["BATCH_SIZE"],
    shuffle=False
)

@torch.no_grad()
def predict_full(model,loader,mode):
    model.eval()

    ys=[]
    ps=[]

    for x,q,y,_,_ in loader:
        x=x.to(DEVICE)
        q=q.to(DEVICE)

        if mode=="traffic":
            p=model(x)

        elif mode=="concat":
            p=model(x,q)

        elif mode=="vra":
            p=model(x,q)

        else:
            raise ValueError(mode)

        ys.append(y.numpy())
        ps.append(p.cpu().numpy())

    y=np.concatenate(ys)
    p=np.concatenate(ps)

    return (
        inverse_target(y),
        inverse_target(p)
    )

def metric_rows(y,p,name):
    rows=[]

    for j in range(NO):
        yt=y[:,j]
        yp=p[:,j]

        rows.append({
            "Model":name,
            "Horizon":f"{30*(j+1)}s",
            "MAE":mean_absolute_error(yt,yp),
            "RMSE":mean_squared_error(yt,yp)**0.5,
            "WAPE_%":(
                np.abs(yt-yp).sum()
                / max(np.abs(yt).sum(),1e-9)
                *100
            ),
            "R2":r2_score(yt,yp),
        })

    return pd.DataFrame(rows)

rr_backbone = TransformerModel().to(DEVICE)
rr_backbone.load_state_dict(rr_backbone_state)

models = {
    "Transformer":(transformer,"traffic"),
    "ReliabilityConcat":(reliability_concat,"concat"),
    "VRA-TFormer":(vra,"vra"),
    "RR-TFormer":(rr_backbone,"traffic"),
    "RB-TFormer":(rb,"traffic"),
}

clean_store={}
clean_tabs=[]

for name,(model,mode) in models.items():
    y,p = predict_full(
        model,
        test_loader,
        mode
    )

    clean_store[name]=(y,p)

    clean_tabs.append(
        metric_rows(
            y,p,name
        )
    )

clean_results=pd.concat(
    clean_tabs,
    ignore_index=True
)

clean_mean=(
    clean_results
    .groupby("Model",as_index=False)
    .agg(
        MAE=("MAE","mean"),
        RMSE=("RMSE","mean"),
        WAPE_pct=("WAPE_%","mean"),
        R2=("R2","mean"),
    )
    .sort_values("MAE")
)

display(clean_results)
display(clean_mean)

In [ ]:
# ============================================================
# 9. PAIRED REAL DEGRADED-CACHE ROBUSTNESS
# ============================================================
ROBUST_L = CFG["ROBUST_LOOKBACK_STEPS"]

degraded_sequences = sorted(
    degraded_df["sequence"].unique()
)

seq_common_length={}

for seq in degraded_sequences:
    clean_bins=set(
        clean_df.loc[
            clean_df["sequence"]==seq,
            "bin_idx"
        ].astype(int)
    )

    lengths=[]

    for (_, _),g in (
        degraded_df[
            degraded_df["sequence"]==seq
        ]
        .groupby(["condition","severity"])
    ):
        deg_bins=set(
            g["bin_idx"].astype(int)
        )

        lengths.append(
            len(clean_bins & deg_bins)
        )

    if lengths:
        seq_common_length[seq]=min(lengths)

robust_sequences=[
    s
    for s,n in seq_common_length.items()
    if n >= ROBUST_L + FUTURE + 1
]

print("Robustness sequences:",robust_sequences)

def prep_frame(df0):
    g=(
        df0
        .sort_values("bin_idx")
        .drop_duplicates("bin_idx")
        .copy()
    )

    for c in REL_COLS:
        g[c]=(
            g[c]
            .interpolate(limit_direction="both")
            .fillna(rel_median[c])
        )

    return g

def build_pair(scenario_df,seq):
    deg=prep_frame(
        scenario_df[
            scenario_df["sequence"]==seq
        ]
    )

    ref=prep_frame(
        clean_df[
            clean_df["sequence"]==seq
        ]
    )

    common=sorted(
        set(deg["bin_idx"].astype(int))
        & set(ref["bin_idx"].astype(int))
    )

    if not common:
        return [],[]

    deg=(
        deg[
            deg["bin_idx"].isin(common)
        ]
        .sort_values("bin_idx")
        .reset_index(drop=True)
    )

    ref=(
        ref[
            ref["bin_idx"].isin(common)
        ]
        .sort_values("bin_idx")
        .reset_index(drop=True)
    )

    n=min(len(deg),len(ref))

    deg=deg.iloc[:n].copy()
    ref=ref.iloc[:n].copy()

    xd=x_scaler.transform(
        deg[TRAFFIC_COLS]
    ).astype(np.float32)

    qd=q_scaler.transform(
        deg[REL_COLS]
    ).astype(np.float32)

    xc=x_scaler.transform(
        ref[TRAFFIC_COLS]
    ).astype(np.float32)

    qc=q_scaler.transform(
        ref[REL_COLS]
    ).astype(np.float32)

    clean_total=ref["total"].values.astype(float)

    dr=[]
    cr=[]

    for t in range(
        ROBUST_L,
        n-FUTURE+1
    ):
        y_raw=np.array([
            clean_total[t+h*B:t+(h+1)*B].sum()
            for h in range(H)
        ],dtype=np.float32)

        y=(
            y_scaler
            .transform(
                y_raw.reshape(-1,1)
            )
            .reshape(-1)
            .astype(np.float32)
        )

        dr.append({
            "x":xd[t-ROBUST_L:t],
            "q":qd[t-ROBUST_L:t],
            "y":y,
        })

        cr.append({
            "x":xc[t-ROBUST_L:t],
            "q":qc[t-ROBUST_L:t],
            "y":y,
        })

    return dr,cr

class PairDS(Dataset):
    def __init__(self,rows):
        self.rows=rows

    def __len__(self):
        return len(self.rows)

    def __getitem__(self,i):
        r=self.rows[i]

        return (
            torch.tensor(r["x"],dtype=torch.float32),
            torch.tensor(r["q"],dtype=torch.float32),
            torch.tensor(r["y"],dtype=torch.float32),
            "seq",
            i,
        )

def pair_loader(rows):
    return DataLoader(
        PairDS(rows),
        batch_size=CFG["BATCH_SIZE"],
        shuffle=False
    )

robust_rows=[]
robust_store={}

for (
    condition,
    severity
),scenario_df in (
    degraded_df
    .groupby(["condition","severity"])
):
    dr_all=[]
    cr_all=[]

    for seq in robust_sequences:
        dr,cr=build_pair(
            scenario_df,
            seq
        )

        dr_all.extend(dr)
        cr_all.extend(cr)

    if not dr_all:
        continue

    dl=pair_loader(dr_all)
    cl=pair_loader(cr_all)

    print(
        condition,
        severity,
        "windows",
        len(dr_all)
    )

    for name,(model,mode) in models.items():
        yc,pc=predict_full(
            model,
            cl,
            mode
        )

        yd,pd0=predict_full(
            model,
            dl,
            mode
        )

        if not np.allclose(yc,yd):
            raise RuntimeError(
                "Paired target mismatch."
            )

        clean_mae=float(
            np.abs(yc-pc).mean()
        )

        deg_mae=float(
            np.abs(yd-pd0).mean()
        )

        rd=(
            (deg_mae-clean_mae)
            / max(clean_mae,1e-9)
            *100
        )

        robust_rows.append({
            "Model":name,
            "Condition":condition,
            "Severity":int(severity),
            "Clean_MAE":clean_mae,
            "Degraded_MAE":deg_mae,
            "Relative_Degradation_%":rd,
            "N_windows":len(dr_all),
            "N_sequences":len(robust_sequences),
        })

        robust_store[
            (
                condition,
                int(severity),
                name
            )
        ]={
            "y":yd,
            "pred":pd0,
            "window_error":(
                np.abs(yd-pd0)
                .mean(axis=1)
            ),
        }

robust_summary=pd.DataFrame(
    robust_rows
)

avg_robust=(
    robust_summary
    .groupby("Model",as_index=False)
    .agg(
        Clean_MAE=("Clean_MAE","mean"),
        Degraded_MAE=("Degraded_MAE","mean"),
        Relative_Degradation_pct=(
            "Relative_Degradation_%",
            "mean"
        ),
        Worst_Scenario_RD=(
            "Relative_Degradation_%",
            "max"
        ),
    )
    .sort_values("Degraded_MAE")
)

display(robust_summary)
display(avg_robust)

In [ ]:
# ============================================================
# 10. STATISTICAL COMPARISON: RB-TFORMER
# ============================================================
clean_stat_rows=[]

y_rb,p_rb = clean_store["RB-TFormer"]

for comparator in [
    "Transformer",
    "ReliabilityConcat",
    "VRA-TFormer",
    "RR-TFormer",
]:
    y_b,p_b=clean_store[comparator]

    temp=[]
    pvals=[]

    for j in range(NO):
        e_rb=np.abs(
            y_rb[:,j]-p_rb[:,j]
        )

        e_b=np.abs(
            y_b[:,j]-p_b[:,j]
        )

        try:
            stat,p=wilcoxon(
                e_rb,
                e_b,
                alternative="less"
            )
        except Exception:
            stat,p=np.nan,1.0

        temp.append({
            "Comparator":comparator,
            "Horizon":f"{30*(j+1)}s",
            "Comparator_MAE":e_b.mean(),
            "RB_MAE":e_rb.mean(),
            "RB_Improvement_%":(
                (e_b.mean()-e_rb.mean())
                / max(e_b.mean(),1e-9)
                *100
            ),
            "RB_win_rate_%":(
                np.mean(e_rb<e_b)
                *100
            ),
            "Wilcoxon_stat":stat,
            "p_raw":p,
        })

        pvals.append(p)

    reject,padj,_,_=multipletests(
        pvals,
        alpha=0.05,
        method="holm"
    )

    for row,pa,rj in zip(
        temp,
        padj,
        reject
    ):
        row["p_holm"]=pa
        row["significant_0.05"]=rj
        clean_stat_rows.append(row)

clean_stats=pd.DataFrame(
    clean_stat_rows
)

robust_stat_rows=[]

scenarios=(
    robust_summary[
        ["Condition","Severity"]
    ]
    .drop_duplicates()
    .sort_values(
        ["Condition","Severity"]
    )
)

for condition,severity in scenarios.itertuples(
    index=False,
    name=None
):
    rb_err=robust_store[
        (
            condition,
            int(severity),
            "RB-TFormer"
        )
    ]["window_error"]

    for comparator in [
        "Transformer",
        "ReliabilityConcat",
        "VRA-TFormer",
        "RR-TFormer",
    ]:
        be=robust_store[
            (
                condition,
                int(severity),
                comparator
            )
        ]["window_error"]

        try:
            stat,p=wilcoxon(
                rb_err,
                be,
                alternative="less"
            )
        except Exception:
            stat,p=np.nan,1.0

        robust_stat_rows.append({
            "Condition":condition,
            "Severity":int(severity),
            "Comparator":comparator,
            "Comparator_MAE":be.mean(),
            "RB_MAE":rb_err.mean(),
            "RB_Improvement_%":(
                (be.mean()-rb_err.mean())
                / max(be.mean(),1e-9)
                *100
            ),
            "RB_win_rate_%":(
                np.mean(rb_err<be)
                *100
            ),
            "p_raw":p,
            "Wilcoxon_stat":stat,
        })

robust_stats=pd.DataFrame(
    robust_stat_rows
)

pieces=[]

for comparator,g in robust_stats.groupby(
    "Comparator"
):
    g=g.copy()

    reject,padj,_,_=multipletests(
        g["p_raw"].values,
        alpha=0.05,
        method="holm"
    )

    g["p_holm"]=padj
    g["significant_0.05"]=reject

    pieces.append(g)

robust_stats=pd.concat(
    pieces,
    ignore_index=True
)

display(clean_stats)
display(robust_stats)

In [ ]:
# ============================================================
# 11. FINAL DECISION
# ============================================================
clean_lookup=clean_mean.set_index("Model")
rob_lookup=avg_robust.set_index("Model")

rb_clean=float(
    clean_lookup.loc[
        "RB-TFormer",
        "MAE"
    ]
)

rb_deg=float(
    rob_lookup.loc[
        "RB-TFormer",
        "Degraded_MAE"
    ]
)

rb_rd=float(
    rob_lookup.loc[
        "RB-TFormer",
        "Relative_Degradation_pct"
    ]
)

transformer_clean=float(
    clean_lookup.loc[
        "Transformer",
        "MAE"
    ]
)

transformer_deg=float(
    rob_lookup.loc[
        "Transformer",
        "Degraded_MAE"
    ]
)

concat_clean=float(
    clean_lookup.loc[
        "ReliabilityConcat",
        "MAE"
    ]
)

h120=(
    clean_results[
        clean_results["Horizon"]=="120s"
    ]
    .set_index("Model")["MAE"]
)

rb_120=float(
    h120["RB-TFormer"]
)

# Pre-declared practical decision.
# RB succeeds if it retains near-best clean accuracy AND
# improves absolute degraded MAE over vanilla Transformer.
clean_near_best = (
    rb_clean <= concat_clean*1.01
)

absolute_robust_better = (
    rb_deg < transformer_deg
)

clean_better_transformer = (
    rb_clean < transformer_clean
)

h120_guardrail = (
    rb_120 <= 11.50
)

decision=pd.DataFrame([
    [
        "Clean within 1% of best clean model",
        clean_near_best,
        rb_clean,
        concat_clean*1.01,
    ],
    [
        "Absolute degraded MAE better than Transformer",
        absolute_robust_better,
        rb_deg,
        transformer_deg,
    ],
    [
        "Clean MAE better than Transformer",
        clean_better_transformer,
        rb_clean,
        transformer_clean,
    ],
    [
        "120-s MAE <=11.50",
        h120_guardrail,
        rb_120,
        11.50,
    ],
],columns=[
    "Criterion",
    "Pass",
    "RB_value",
    "Threshold_reference",
])

display(decision)

if (
    clean_near_best
    and absolute_robust_better
    and h120_guardrail
):
    FINAL_DECISION=(
        "RB-TFormer qualifies as the final proposed model: "
        "validation-selected checkpoint balancing retains near-best clean accuracy "
        "and improves absolute degraded forecasting error."
    )

elif (
    clean_better_transformer
    and rb_deg <= transformer_deg*1.01
    and h120_guardrail
):
    FINAL_DECISION=(
        "RB-TFormer provides a defensible clean-robustness Pareto trade-off: "
        "clean forecasting improves over Transformer while degraded absolute error "
        "remains essentially unchanged."
    )

else:
    FINAL_DECISION=(
        "RB-TFormer does not improve the clean-robustness Pareto frontier enough "
        "to justify replacing the existing accuracy-robustness study framing."
    )

print("\n============================================")
print(FINAL_DECISION)
print("Selected alpha:",SELECTED_ALPHA)
print("============================================")

In [ ]:
# ============================================================
# 12. SAVE PAPER-READY OUTPUTS
# ============================================================
OUTDIR=D["rb"]

clean_results.to_csv(
    OUTDIR/"RB_clean_results.csv",
    index=False
)

clean_mean.to_csv(
    OUTDIR/"RB_clean_mean.csv",
    index=False
)

robust_summary.to_csv(
    OUTDIR/"RB_robustness_summary.csv",
    index=False
)

avg_robust.to_csv(
    OUTDIR/"RB_robustness_average.csv",
    index=False
)

clean_stats.to_csv(
    OUTDIR/"RB_clean_wilcoxon.csv",
    index=False
)

robust_stats.to_csv(
    OUTDIR/"RB_robustness_wilcoxon.csv",
    index=False
)

decision.to_csv(
    OUTDIR/"RB_decision.csv",
    index=False
)

xlsx_out=(
    OUTDIR
    / "SmartTraffic_RB_TFormer_Results.xlsx"
)

with pd.ExcelWriter(
    xlsx_out,
    engine="openpyxl"
) as writer:

    split_summary.to_excel(
        writer,
        sheet_name="Split",
        index=False
    )

    selection.to_excel(
        writer,
        sheet_name="Validation_Alpha",
        index=False
    )

    clean_results.to_excel(
        writer,
        sheet_name="Clean_Horizon",
        index=False
    )

    clean_mean.to_excel(
        writer,
        sheet_name="Clean_Mean",
        index=False
    )

    robust_summary.to_excel(
        writer,
        sheet_name="Robustness",
        index=False
    )

    avg_robust.to_excel(
        writer,
        sheet_name="Robust_Average",
        index=False
    )

    clean_stats.to_excel(
        writer,
        sheet_name="Clean_Wilcoxon",
        index=False
    )

    robust_stats.to_excel(
        writer,
        sheet_name="Robust_Wilcoxon",
        index=False
    )

    decision.to_excel(
        writer,
        sheet_name="Decision",
        index=False
    )

# Figure 1 — validation selection curve
fig,ax=plt.subplots(
    figsize=(7.2,4.0)
)

ax.plot(
    selection["alpha"],
    selection["Validation_Clean_MAE"],
    marker="o",
    label="Clean validation"
)

ax.plot(
    selection["alpha"],
    selection["Validation_Corrupt_MAE"],
    marker="o",
    label="Synthetic-corruption validation"
)

ax.axvline(
    SELECTED_ALPHA,
    linestyle="--",
    linewidth=1.0,
    label=f"Selected α={SELECTED_ALPHA:.1f}"
)

ax.set_xlabel("Checkpoint interpolation α")
ax.set_ylabel("MAE")
ax.set_title("Validation-only checkpoint selection")
ax.legend(frameon=False)

fig.tight_layout()

fig.savefig(
    OUTDIR/"RB_validation_alpha_selection.png",
    dpi=CFG["FIG_DPI"],
    bbox_inches="tight"
)

fig.savefig(
    OUTDIR/"RB_validation_alpha_selection.pdf",
    bbox_inches="tight"
)

plt.show()

# Figure 2 — clean horizon
fig,ax=plt.subplots(
    figsize=(7.4,4.0)
)

for name in [
    "Transformer",
    "ReliabilityConcat",
    "VRA-TFormer",
    "RR-TFormer",
    "RB-TFormer",
]:
    z=clean_results[
        clean_results["Model"]==name
    ]

    ax.plot(
        z["Horizon"],
        z["MAE"],
        marker="o",
        label=name
    )

ax.set_xlabel("Forecast horizon")
ax.set_ylabel("MAE (vehicles / 30-s block)")
ax.set_title("Multi-horizon clean forecasting")
ax.legend(frameon=False,ncol=2)

fig.tight_layout()

fig.savefig(
    OUTDIR/"RB_clean_multi_horizon.png",
    dpi=CFG["FIG_DPI"],
    bbox_inches="tight"
)

fig.savefig(
    OUTDIR/"RB_clean_multi_horizon.pdf",
    bbox_inches="tight"
)

plt.show()

# Figure 3 — clean vs degraded absolute MAE
fig,ax=plt.subplots(
    figsize=(7.5,4.0)
)

z=avg_robust.copy()

x=np.arange(len(z))
w=0.35

ax.bar(
    x-w/2,
    z["Clean_MAE"],
    width=w,
    label="Matched clean"
)

ax.bar(
    x+w/2,
    z["Degraded_MAE"],
    width=w,
    label="Degraded"
)

ax.set_xticks(x)
ax.set_xticklabels(
    z["Model"],
    rotation=20
)

ax.set_ylabel("Absolute MAE")
ax.set_title("Paired clean and degraded forecasting error")
ax.legend(frameon=False)

fig.tight_layout()

fig.savefig(
    OUTDIR/"RB_absolute_robustness.png",
    dpi=CFG["FIG_DPI"],
    bbox_inches="tight"
)

fig.savefig(
    OUTDIR/"RB_absolute_robustness.pdf",
    bbox_inches="tight"
)

plt.show()

# Figure 4 — clean/robustness Pareto plot
fig,ax=plt.subplots(
    figsize=(6.5,4.8)
)

for _,r in avg_robust.iterrows():
    clean_val=float(
        clean_lookup.loc[
            r["Model"],
            "MAE"
        ]
    )

    ax.scatter(
        clean_val,
        r["Degraded_MAE"],
        s=55
    )

    ax.annotate(
        r["Model"],
        (
            clean_val,
            r["Degraded_MAE"]
        ),
        xytext=(5,4),
        textcoords="offset points",
        fontsize=8
    )

ax.set_xlabel("Clean test MAE")
ax.set_ylabel("Degraded absolute MAE")
ax.set_title("Clean–robustness Pareto comparison")

fig.tight_layout()

fig.savefig(
    OUTDIR/"RB_clean_robustness_pareto.png",
    dpi=CFG["FIG_DPI"],
    bbox_inches="tight"
)

fig.savefig(
    OUTDIR/"RB_clean_robustness_pareto.pdf",
    bbox_inches="tight"
)

plt.show()

digest=[
    "# SmartTraffic RB-TFormer — Final Validation-Selected Interpolation Digest",
    "",
    "## Model naming",
    "- Transformer",
    "- ReliabilityConcat",
    "- VRA-TFormer",
    "- RR-TFormer",
    "- RB-TFormer",
    "",
    "## Selection protocol",
    "- RB-TFormer alpha is selected only from validation data.",
    "- Selection combines clean validation MAE and synthetic-corruption validation MAE.",
    "- Clean test and real degraded cache are untouched until alpha is locked.",
    "",
    f"## Selected alpha\n- **α = {SELECTED_ALPHA:.1f}**",
    "",
    "## Validation selection",
    selection.to_markdown(index=False),
    "",
    "## Clean mean results",
    clean_mean.to_markdown(index=False),
    "",
    "## Average robustness",
    avg_robust.to_markdown(index=False),
    "",
    "## Decision criteria",
    decision.to_markdown(index=False),
    "",
    "## Final decision",
    FINAL_DECISION,
]

digest_path=(
    OUTDIR
    / "SmartTraffic_RB_TFormer_Digest.md"
)

digest_path.write_text(
    "\n".join(digest),
    encoding="utf-8"
)

zip_out=(
    OUTDIR
    / "SmartTraffic_RB_TFormer_Artifacts.zip"
)

with zipfile.ZipFile(
    zip_out,
    "w",
    zipfile.ZIP_DEFLATED
) as zf:

    for p in OUTDIR.iterdir():
        if (
            p.is_file()
            and p.name!=zip_out.name
        ):
            zf.write(
                p,
                arcname=p.name
            )

print("\n============================================")
print("RB-TFORMER EXPERIMENT COMPLETE")
print("============================================")
print("Excel :",xlsx_out)
print("Digest:",digest_path)
print("ZIP   :",zip_out)
print("\nUPLOAD THESE THREE FILES BACK TO CHAT.")